# Homework 2, Part 1: Data Cleaning and Validation

## 1.1 Load and inspect the data

This section loads the delivered `ohlc_hw` table and describes it before any repair. The database connection is read-only. Rows are sorted by ticker and date on the way in, so later return and fill calculations start from a stable order. Ticker normalization, the data-quality report, and the cleaned table `ohlc_hw_cleaned` are later sections.

The course database belongs at `Lectures/Lecture 4/data/data.db`. Section 1.3 writes `ohlc_hw_cleaned` only to the working copy at `Homeworks/HW2/Dhruv_Gupta/data/data.db`. That copy is created from the database this notebook read when the copy does not already exist.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)


def find_repo_root() -> Path:
    """Walk upward from the working directory to the course repository."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "Homeworks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the course repository. "
        "Could not find pyproject.toml above the working directory."
    )


REPO_ROOT = find_repo_root()
HOMEWORK_DIR = REPO_ROOT / "Homeworks" / "HW2" / "Dhruv_Gupta"
CANDIDATE_DBS = [
    HOMEWORK_DIR / "data" / "data.db",
    REPO_ROOT / "Lectures" / "Lecture 4" / "data" / "data.db",
    REPO_ROOT / "data.db",
]
db_path = next((path for path in CANDIDATE_DBS if path.is_file()), None)
if db_path is None:
    searched = "\n".join(str(path) for path in CANDIDATE_DBS)
    raise FileNotFoundError(f"Download data.db from bCourses. Looked in:\n{searched}")

print(f"Reading {db_path}")
print("Read-only connection. This section does not write ohlc_hw_cleaned.")

Reading /Users/dhruvgupta/Desktop/PythonPreProgramBerkeleyMFE/UCB-MFE-python-preprogram-2026-winter/Lectures/Lecture 4/data/data.db
Read-only connection. This section does not write ohlc_hw_cleaned.


### Load `ohlc_hw`

Each row is one raw ticker label on one observation date. The date column in this table is `ts`. Lecture examples that call the same field `date` need an explicit rename before those functions are reused. `ohlc` is a separate reference table and is not the homework delivery.

In [2]:
conn = sqlite3.connect(db_path.resolve().as_uri() + "?mode=ro", uri=True)
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
    conn,
)
display(tables)

ohlc_hw = pd.read_sql_query(
    """
    SELECT ts, ticker, open, high, low, close, adj_close, volume
    FROM ohlc_hw
    ORDER BY ticker, ts
    """,
    conn,
    parse_dates=["ts"],
)
conn.close()

ohlc_hw = ohlc_hw.sort_values(["ticker", "ts"], kind="mergesort").reset_index(drop=True)
print(f"Loaded {len(ohlc_hw):,} rows and {ohlc_hw.shape[1]} columns from ohlc_hw.")

,name
0,market_caps
1,ohlc
2,ohlc_hw


Loaded 397,711 rows and 8 columns from ohlc_hw.


### Basic statistics

`.info()` shows the column types and how many values are present. `.describe()` summarizes the numeric price and volume columns. These figures describe the raw delivery, including whatever missing values and repeated labels it contains.

In [3]:
ohlc_hw.info()
display(ohlc_hw.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   ticker     397711 non-null  object        
 2   open       397711 non-null  float64       
 3   high       397711 non-null  float64       
 4   low        397711 non-null  float64       
 5   close      397711 non-null  float64       
 6   adj_close  377856 non-null  float64       
 7   volume     397711 non-null  int64         
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292486272,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


### Column meanings

The preview prints each ticker with `repr` so HTML markup stays visible as text. A normal notebook display would render those strings as links and hide the formatting problem.

In [4]:
column_meanings = pd.DataFrame(
    [
        ("ts", "Observation date. Lecture examples call this field date; this table names it ts."),
        ("ticker", "Security label as stored. It may contain markup, whitespace, or inconsistent case."),
        ("open", "Raw opening price for that label and date."),
        ("high", "Raw high price."),
        ("low", "Raw low price."),
        ("close", "Raw closing price. Part 1 return checks use this column."),
        ("adj_close", "Vendor adjusted close. A missing adjusted close is separate from a missing raw close."),
        ("volume", "Reported share volume. An empty volume is not the same fact as a recorded zero."),
    ],
    columns=["column", "meaning"],
)
display(column_meanings)

preview = ohlc_hw.head(8).copy()
preview["ts"] = preview["ts"].dt.strftime("%Y-%m-%d")
preview["ticker"] = preview["ticker"].map(repr)
display(preview)

,column,meaning
0,ts,Observation date. Lecture examples call this f...
1,ticker,Security label as stored. It may contain marku...
2,open,Raw opening price for that label and date.
3,high,Raw high price.
4,low,Raw low price.
5,close,Raw closing price. Part 1 return checks use th...
6,adj_close,Vendor adjusted close. A missing adjusted clos...
7,volume,Reported share volume. An empty volume is not ...


,ts,ticker,open,high,low,close,adj_close,volume
0,1981-10-19,"'<a target=""_blank""><span style=""color: blue"">...",0.083147,0.083147,0.083147,0.083147,0.064096,20585620
1,1981-10-20,"'<a target=""_blank""><span style=""color: blue"">...",0.087612,0.087612,0.087612,0.087612,0.067538,35728032
2,1982-01-20,"'<a target=""_blank""><span style=""color: blue"">...",0.090402,0.090402,0.090402,0.090402,0.069689,25827224
3,1982-02-11,"'<a target=""_blank""><span style=""color: blue"">...",0.082589,0.082589,0.082589,0.082589,0.063666,24528024
4,1982-03-26,"'<a target=""_blank""><span style=""color: blue"">...",0.072545,0.072545,0.072545,0.072545,0.055923,50780848
5,1982-05-03,"'<a target=""_blank""><span style=""color: blue"">...",0.068080,0.068080,0.068080,0.068080,0.052481,82700880
6,1982-06-04,"'<a target=""_blank""><span style=""color: blue"">...",0.058594,0.058594,0.058594,0.058594,0.045169,37676836
7,1983-03-02,"'<a target=""_blank""><span style=""color: blue"">...",0.207030,0.209820,0.206470,0.208710,0.160890,105952103


### Date range and ticker labels

The range below is the earliest and latest `ts` in the whole table. Individual labels do not all cover that full span. The ticker table lists every raw label, still unnormalized. The same symbol can appear more than once because the stored text differs. Section 1.2 will normalize those labels before any grouping or duplicate check.

In [5]:
first_date = ohlc_hw["ts"].min()
last_date = ohlc_hw["ts"].max()
n_labels = ohlc_hw["ticker"].nunique(dropna=False)
print(f"Date range: {first_date.date()} through {last_date.date()}")
print(f"Raw ticker labels: {n_labels}")

ticker_summary = (
    ohlc_hw.groupby("ticker", dropna=False)
    .agg(rows=("ts", "size"), first_date=("ts", "min"), last_date=("ts", "max"))
    .reset_index()
    .sort_values("ticker", kind="mergesort")
    .reset_index(drop=True)
)
ticker_summary["ticker"] = ticker_summary["ticker"].map(repr)
ticker_summary["first_date"] = ticker_summary["first_date"].dt.strftime("%Y-%m-%d")
ticker_summary["last_date"] = ticker_summary["last_date"].dt.strftime("%Y-%m-%d")
display(ticker_summary)

Date range: 1962-01-02 through 2024-12-06
Raw ticker labels: 150


,ticker,rows,first_date,last_date
0,"'<a target=""_blank""><span style=""color: blue"">...",307,1981-10-19,2024-09-04
1,"'<a target=""_blank""><span style=""color: blue"">...",294,1983-07-29,2024-08-14
2,"'<a target=""_blank""><span style=""color: blue"">...",185,1997-07-02,2024-12-03
3,"'<a target=""_blank""><span style=""color: blue"">...",356,1972-04-14,2024-10-17
4,"'<a target=""_blank""><span style=""color: blue"">...",462,1962-03-23,2024-09-04
5,"'<a target=""_blank""><span style=""color: blue"">...",394,1962-02-08,2024-09-26
6,"'<a target=""_blank""><span style=""color: blue"">...",129,2004-07-01,2023-08-24
7,"'<a target=""_blank""><span style=""color: blue"">...",230,1990-05-22,2024-07-22
8,"'<a target=""_blank""><span style=""color: blue"">...",400,1962-05-10,2024-11-01
9,"'<a target=""_blank""><span style=""color: blue"">...",458,1962-01-08,2024-11-05


## 1.2 Detect data-quality issues

This section follows the Lecture 4 module 1 checks (`missing_sessions` and `price_issue_report`) and the module 3 `quality_flags` split between raw OHLC bounds and adjusted-close validity. The lecture names the date column `date`. This table names it `ts`, so every function below reads `ts` on purpose.

Ticker markup, whitespace, and case are normalized before any grouping, duplicate check, or return. The original labels are kept in an audit. Exact duplicate rows are recorded and then removed, which is the lecture's order. Conflicting `(ticker, ts)` keys are not resolved by keeping the first row.

**Reference calendar.** For each normalized ticker, the expected sessions are the dates in the supplied `ohlc` table that fall inside that ticker's own first and last `ohlc_hw` date. Business weekdays are not used, because they include exchange holidays. A result of zero missing sessions is reported as zero.

**Severity**

| Issue | Severity | Reason |
|---|---|---|
| Missing raw OHLC, impossible OHLC, nonpositive or non-finite price, negative volume, conflicting key | high | The observation cannot be used as printed |
| Missing adjusted close, missing volume, missing session, extreme return, zero volume | medium | Incomplete or suspicious, but not an automatic deletion |
| Exact duplicate, zero or undefined return standard deviation | low | Redundant copy, or the 3σ screen is deliberately not applied |

`quality_report_before_repair` is the diagnosis of the delivered data. Repairs in this section are run only on small injected examples for checks that this database does not contain. The same detector is rerun after each of those repairs. The full-table repair is section 1.3.


In [6]:
import re

import numpy as np

RAW_PRICE_COLUMNS = ["open", "high", "low", "close"]
EQUITY_PRICE_COLUMNS = RAW_PRICE_COLUMNS + ["adj_close"]
VALUE_COLUMNS = ["ticker", "ts", *EQUITY_PRICE_COLUMNS, "volume"]
REPORT_COLUMNS = ["ticker", "issue_type", "issue_date", "description", "severity"]
KEY_COLUMNS = ["ticker", "ts"]

SEVERITY = {
    "missing_price": "high",
    "missing_adj_close": "medium",
    "missing_volume": "medium",
    "missing_session": "medium",
    "extreme_return": "medium",
    "zero_volume": "medium",
    "high_below_close": "high",
    "low_above_close": "high",
    "ohlc_bounds": "high",
    "nonpositive_price": "high",
    "nonfinite_price": "high",
    "invalid_volume": "high",
    "exact_duplicate": "low",
    "conflicting_key": "high",
    "undefined_return_screen": "low",
}


def normalize_ticker(value: object) -> str:
    """Remove HTML markup, then apply the lecture's strip-and-uppercase rule."""
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"<[^>]+>", "", text)
    return " ".join(text.split()).upper()


def empty_report() -> pd.DataFrame:
    return pd.DataFrame(columns=REPORT_COLUMNS)


def concat_issues(parts: list[pd.DataFrame]) -> pd.DataFrame:
    kept = [part for part in parts if part is not None and not part.empty]
    if not kept:
        return empty_report()
    report = pd.concat(kept, ignore_index=True)
    return (
        report[REPORT_COLUMNS]
        .sort_values(["ticker", "issue_date", "issue_type"], kind="mergesort")
        .reset_index(drop=True)
    )


def issue_rows(frame: pd.DataFrame, mask: pd.Series, issue_type: str, description: str) -> pd.DataFrame:
    """One report row per flagged observation. An empty mask returns an empty report."""
    rows = frame.loc[mask, KEY_COLUMNS].rename(columns={"ts": "issue_date"}).copy()
    rows["issue_type"] = issue_type
    rows["description"] = description
    rows["severity"] = SEVERITY[issue_type]
    return rows[REPORT_COLUMNS]


def attach_labels(frame: pd.DataFrame) -> pd.DataFrame:
    """Preserve the delivered ticker text and sort by normalized ticker and date."""
    labeled = frame.copy()
    labeled["ticker_raw"] = labeled["ticker"].astype("string")
    labeled["ticker"] = labeled["ticker_raw"].map(normalize_ticker)
    labeled["ts"] = pd.to_datetime(labeled["ts"])
    return labeled.sort_values(["ticker", "ts", "ticker_raw"], kind="mergesort").reset_index(drop=True)


def split_duplicate_keys(labeled: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Separate exact copies from conflicting keys.

    Exact copies are safe to collapse. Conflicting copies stay out of the return
    screen because choosing the first row would hide the disagreement.
    """
    analysis = labeled[VALUE_COLUMNS].copy()
    exact_mask = analysis.duplicated(VALUE_COLUMNS, keep=False)
    exact_rows = analysis.loc[exact_mask].copy()
    deduped = analysis.drop_duplicates(VALUE_COLUMNS).reset_index(drop=True)
    conflict_mask = deduped.duplicated(KEY_COLUMNS, keep=False)
    conflicts = deduped.loc[conflict_mask].copy()
    diagnosed = deduped.loc[~conflict_mask].sort_values(KEY_COLUMNS, kind="mergesort").reset_index(drop=True)
    return exact_rows, conflicts, diagnosed, analysis


def missing_sessions(frame: pd.DataFrame, expected_dates) -> pd.DatetimeIndex:
    """Return expected dates absent from one ticker's observations.

    Adapted from Lecture 4 module 1. The lecture column is date; this table uses ts.
    The returned index contains dates only. It does not invent prices.
    """
    observed = pd.DatetimeIndex(frame["ts"].dropna().unique())
    return pd.DatetimeIndex(expected_dates).difference(observed).sort_values()


def session_gap_report(observations: pd.DataFrame, reference: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare each ticker with its own ohlc dates over its supplied history."""
    span = (
        observations.groupby("ticker", dropna=False)["ts"]
        .agg(history_start="min", history_end="max")
        .reset_index()
    )
    calendar = reference[["ticker", "ts"]].dropna().drop_duplicates()
    calendar["ticker"] = calendar["ticker"].map(normalize_ticker)
    calendar["ts"] = pd.to_datetime(calendar["ts"])
    expected = calendar.merge(span, on="ticker", how="inner")
    expected = expected.loc[
        expected["ts"].ge(expected["history_start"]) & expected["ts"].le(expected["history_end"])
    ]
    observed = observations[KEY_COLUMNS].drop_duplicates()
    comparison = expected.merge(observed, on=KEY_COLUMNS, how="left", indicator=True)
    absent = comparison.loc[comparison["_merge"].eq("left_only"), KEY_COLUMNS].copy()
    counts = absent.groupby("ticker").size()

    summary = span.sort_values("ticker", kind="mergesort").reset_index(drop=True)
    summary["missing_days"] = summary["ticker"].map(counts).fillna(0).astype(int)
    summary["history_start"] = summary["history_start"].dt.strftime("%Y-%m-%d")
    summary["history_end"] = summary["history_end"].dt.strftime("%Y-%m-%d")
    summary["summary"] = summary.apply(
        lambda row: (
            f"Ticker {row.ticker} is missing {row.missing_days} trading days "
            f"between {row.history_start} and {row.history_end}"
        ),
        axis=1,
    )
    gap_issues = issue_rows(
        absent,
        pd.Series(True, index=absent.index),
        "missing_session",
        "Date is on this ticker's ohlc calendar and inside its supplied history, but no ohlc_hw row exists. No price was invented.",
    )
    return summary, gap_issues


def simple_close_returns(frame: pd.DataFrame) -> pd.Series:
    """Simple close-to-close return, (close - prev_close) / prev_close, within ticker.

    Lecture 4 uses grouped pct_change on strictly positive finite closes. Nonpositive,
    missing, and non-finite closes are excluded, so the first row and a zero previous
    close stay undefined instead of producing a flag.
    """
    valid_close = frame["close"].where(np.isfinite(frame["close"]) & frame["close"].gt(0))
    previous = valid_close.groupby(frame["ticker"]).shift(1)
    returns = (valid_close - previous) / previous
    return returns.where(np.isfinite(returns))


def return_screen_status(frame: pd.DataFrame, returns: pd.Series) -> pd.DataFrame:
    """Say whether the 3σ screen was applied. Zero and undefined std create no flags."""
    grouped = returns.groupby(frame["ticker"])
    screen = pd.DataFrame({
        "n_defined": returns.notna().groupby(frame["ticker"]).sum().astype(int),
        "std": grouped.std(ddof=1),
    })
    bounds = frame.groupby("ticker")["ts"].agg(history_start="min", history_end="max")
    screen = bounds.join(screen)
    undefined = screen["n_defined"].lt(2) | screen["std"].isna()
    screen["status"] = np.where(
        undefined,
        "undefined_std",
        np.where(screen["std"].eq(0), "zero_std", "applied"),
    )
    return screen.reset_index()


def screen_issue_rows(frame: pd.DataFrame, screen: pd.DataFrame) -> pd.DataFrame:
    skipped = screen.loc[screen["status"].ne("applied")].copy()
    if skipped.empty:
        return empty_report()
    rows = skipped.rename(columns={"history_end": "issue_date"})
    rows["issue_type"] = "undefined_return_screen"
    rows["description"] = rows.apply(
        lambda row: (
            "Sample standard deviation is zero, so no 3σ flags were raised."
            if row.status == "zero_std"
            else "Fewer than two defined simple returns, so the sample standard deviation is undefined and no 3σ flags were raised."
        ),
        axis=1,
    )
    rows["description"] = rows.apply(
        lambda row: (
            f"{row.description} Window {pd.Timestamp(row.history_start).date()} through {pd.Timestamp(row.issue_date).date()}."
        ),
        axis=1,
    )
    rows["severity"] = SEVERITY["undefined_return_screen"]
    return rows[REPORT_COLUMNS]


def price_issue_report(frame: pd.DataFrame) -> pd.DataFrame:
    """Diagnose unique ticker/date rows. Extreme returns are a review flag, not a deletion rule."""
    data = frame.sort_values(KEY_COLUMNS, kind="mergesort").reset_index(drop=True)
    returns = simple_close_returns(data)
    grouped = returns.groupby(data["ticker"])
    mean = grouped.transform("mean")
    std = grouped.transform("std")  # pandas sample standard deviation, ddof=1
    extreme = returns.notna() & std.gt(0) & (returns - mean).abs().gt(3 * std)

    # Lecture 4 module 1 bounds: low <= open <= high and low <= close <= high, plus high < low.
    # Adjusted close is intentionally excluded. Module 3 checks it only for missing, non-finite, and nonpositive values.
    raw_bounds = (
        data["high"].lt(data[["open", "close"]].max(axis=1))
        | data["low"].gt(data[["open", "close"]].min(axis=1))
        | data["high"].lt(data["low"])
    )
    parts = [
        issue_rows(data, data["high"].lt(data["close"]), "high_below_close", "High is below close, which is impossible for a raw bar."),
        issue_rows(data, data["low"].gt(data["close"]), "low_above_close", "Low is above close, which is impossible for a raw bar."),
        issue_rows(data, raw_bounds.fillna(False), "ohlc_bounds", "Raw bar violates low <= open <= high or low <= close <= high."),
        issue_rows(data, data["volume"].eq(0), "zero_volume", "Volume is zero. This can indicate a stale print and is kept for review."),
        issue_rows(
            data,
            extreme.fillna(False),
            "extreme_return",
            "Simple close return is more than 3 sample standard deviations from this ticker's mean. Review flag only; the row is not deleted.",
        ),
        screen_issue_rows(data, return_screen_status(data, returns)),
    ]
    for column in EQUITY_PRICE_COLUMNS:
        values = data[column]
        finite = pd.Series(np.isfinite(values.to_numpy(dtype=float, na_value=np.nan)), index=values.index)
        if column in RAW_PRICE_COLUMNS:
            missing_type = "missing_price"
            missing_text = f"{column} is missing."
        else:
            missing_type = "missing_adj_close"
            missing_text = "adj_close is missing. This is a missing cell, not a missing trading day, and raw OHLC bounds are not applied to it."
        parts.append(issue_rows(data, values.isna(), missing_type, missing_text))
        parts.append(issue_rows(data, values.notna() & ~finite, "nonfinite_price", f"{column} is not finite."))
        parts.append(issue_rows(data, finite & values.le(0), "nonpositive_price", f"{column} is nonpositive."))
    volume_missing = data["volume"].isna()
    volume_invalid = data["volume"].notna() & (~np.isfinite(data["volume"]) | data["volume"].lt(0))
    parts.append(issue_rows(data, volume_missing, "missing_volume", "volume is missing. Unknown volume is not evidence of zero trading."))
    parts.append(issue_rows(data, volume_invalid, "invalid_volume", "volume is negative or non-finite."))
    return concat_issues(parts)


def duplicate_issue_rows(exact_rows: pd.DataFrame, conflicts: pd.DataFrame) -> pd.DataFrame:
    parts = []
    if not exact_rows.empty:
        parts.append(issue_rows(
            exact_rows,
            pd.Series(True, index=exact_rows.index),
            "exact_duplicate",
            "Exact duplicate after ticker normalization. Extra copies are removed before return and price checks.",
        ))
    if not conflicts.empty:
        parts.append(issue_rows(
            conflicts,
            pd.Series(True, index=conflicts.index),
            "conflicting_key",
            "The same normalized ticker and date disagree. The first row is not kept automatically.",
        ))
    return concat_issues(parts)


def repair_invalid_closes(frame: pd.DataFrame) -> pd.DataFrame:
    """Apply the Lecture 4 close repair on a small example.

    An invalid close becomes missing. The previous close replaces it when that previous
    close sits inside today's high and low. Otherwise the midpoint of today's high and
    low is used. A value that still cannot be repaired stays missing and is quarantined.
    Adjusted close is not forced into the raw range and is not copied from the raw close.
    Unknown or negative volume becomes 0 with a repair flag. Extreme returns are left unchanged.
    """
    repaired = frame.sort_values(KEY_COLUMNS, kind="mergesort").reset_index(drop=True).copy()
    bad_close = (
        ~np.isfinite(repaired["close"])
        | repaired["close"].le(0)
        | repaired["close"].lt(repaired["low"])
        | repaired["close"].gt(repaired["high"])
    )
    repaired["close_repaired"] = bad_close.fillna(False)
    repaired.loc[bad_close, "close"] = np.nan
    previous = repaired.groupby("ticker")["close"].ffill()
    within_range = previous.between(repaired["low"], repaired["high"])
    midpoint = (repaired["high"] + repaired["low"]) / 2
    replacement = previous.where(within_range, midpoint)
    repaired["close"] = repaired["close"].where(repaired["close"].notna(), replacement)
    repaired["close_quarantined"] = repaired["close_repaired"] & repaired["close"].isna()

    bad_adjusted = ~np.isfinite(repaired["adj_close"]) | repaired["adj_close"].le(0)
    repaired["adj_close_quarantined"] = bad_adjusted.fillna(False)
    repaired.loc[bad_adjusted, "adj_close"] = np.nan

    invalid_volume = ~np.isfinite(repaired["volume"]) | repaired["volume"].lt(0)
    repaired["volume_repaired"] = invalid_volume.fillna(False)
    repaired.loc[invalid_volume, "volume"] = 0
    return repaired


### Normalize labels, then separate duplicate keys

The delivered labels include plain symbols and the same symbols wrapped in HTML. Normalization happens before duplicate detection. The next cells diagnose `diagnosed`, which contains one row per non-conflicting `(ticker, ts)` after exact copies are removed.


In [7]:
labeled = attach_labels(ohlc_hw)
label_audit = (
    labeled.groupby(["ticker_raw", "ticker"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["ticker", "rows"], kind="mergesort")
    .reset_index(drop=True)
)
print(
    f"Raw ticker labels: {label_audit['ticker_raw'].nunique()} | "
    f"Normalized tickers: {label_audit['ticker'].nunique()}"
)
label_preview = label_audit.head(6).copy()
label_preview["ticker_raw"] = label_preview["ticker_raw"].map(repr)
display(label_preview)

exact_rows, conflicts, diagnosed, analysis = split_duplicate_keys(labeled)
print(f"Rows before removing exact duplicates: {len(analysis):,}")
print(f"Exact-duplicate rows, counting every copy: {len(exact_rows):,}")
print(f"Extra exact copies removed: {int(analysis.duplicated(VALUE_COLUMNS).sum()):,}")
print(f"Conflicting (ticker, ts) rows: {len(conflicts):,}")
print(f"Rows passed to the price and return checks: {len(diagnosed):,}")
if conflicts.empty:
    assert not diagnosed.duplicated(KEY_COLUMNS).any()
    print("No conflicting keys. The lecture assertion holds: each diagnosed key is unique.")
else:
    print("Conflicting records need a source-selection rule. They are excluded from the return screen.")
    conflict_preview = conflicts.head(10).copy()
    display(conflict_preview)


Raw ticker labels: 150 | Normalized tickers: 30


,ticker_raw,ticker,rows
0,"'<a target=""_blank""><span style=""font-weight: ...",AAPL,264
1,"'<a target=""_blank""><span style=""font-weight: ...",AAPL,306
2,"'<a target=""_blank""><span style=""color: blue"">...",AAPL,307
3,"'<a target=""_blank""><span style=""color: red"">A...",AAPL,318
4,'AAPL',AAPL,10747
5,"'<a target=""_blank""><span style=""font-weight: ...",AMGN,272


Rows before removing exact duplicates: 397,711
Exact-duplicate rows, counting every copy: 58,920
Extra exact copies removed: 29,460
Conflicting (ticker, ts) rows: 0
Rows passed to the price and return checks: 368,251


No conflicting keys. The lecture assertion holds: each diagnosed key is unique.


### Missing cells and missing sessions

`.isna()` only sees columns on rows that exist. A missing trading day has no row, so it is found by comparing dates with `missing_sessions`. The summary includes tickers with zero gaps.


In [8]:
column_missing = diagnosed.isna().sum().rename("missing_cells").to_frame()
display(column_missing)

reference_conn = sqlite3.connect(db_path.resolve().as_uri() + "?mode=ro", uri=True)
reference_dates = pd.read_sql_query("SELECT ticker, ts FROM ohlc", reference_conn, parse_dates=["ts"])
reference_conn.close()

gap_summary, gap_issues = session_gap_report(diagnosed, reference_dates)
display(gap_summary[["ticker", "missing_days", "history_start", "history_end", "summary"]])
print(f"Missing sessions in the delivered data: {int(gap_summary['missing_days'].sum())}")


,missing_cells
ticker,0
ts,0
open,0
high,0
low,0
close,0
adj_close,18374
volume,0


,ticker,missing_days,history_start,history_end,summary
0,AAPL,0,1980-12-12,2024-10-18,Ticker AAPL is missing 0 trading days between ...
1,AMGN,0,1983-06-17,2024-10-18,Ticker AMGN is missing 0 trading days between ...
2,AMZN,0,1997-05-15,2024-12-06,Ticker AMZN is missing 0 trading days between ...
3,AXP,0,1972-04-03,2024-10-18,Ticker AXP is missing 0 trading days between 1...
4,BA,0,1962-01-02,2024-10-18,Ticker BA is missing 0 trading days between 19...
5,CAT,0,1962-01-02,2024-10-18,Ticker CAT is missing 0 trading days between 1...
6,CRM,0,2004-06-23,2024-10-18,Ticker CRM is missing 0 trading days between 2...
7,CSCO,0,1990-02-16,2024-10-18,Ticker CSCO is missing 0 trading days between ...
8,CVX,0,1962-01-02,2024-12-06,Ticker CVX is missing 0 trading days between 1...
9,DIS,0,1962-01-02,2024-12-06,Ticker DIS is missing 0 trading days between 1...


Missing sessions in the delivered data: 0


### Injected gap

The delivered data can have zero gaps and still need a working detector. This is the lecture's example: the calendar contains January 2 through January 5, January 3 is removed, and January 1 is never expected. The detector returns the missing date and no price.


In [9]:
expected_sessions = pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"])
calendar_delivery = pd.DataFrame({
    "ticker": ["DEMO_A"] * 3,
    "ts": expected_sessions[[0, 2, 3]],
})
removed_dates = missing_sessions(calendar_delivery, expected_sessions)
print(
    f"DEMO_A is missing {len(removed_dates)} trading days between "
    f"{expected_sessions.min().date()} and {expected_sessions.max().date()}: "
    f"{[date.date() for date in removed_dates]}"
)
assert removed_dates.tolist() == [pd.Timestamp("2024-01-03")]
injected_gap = pd.DataFrame({
    "ticker": ["DEMO_A"],
    "issue_type": ["missing_session"],
    "issue_date": removed_dates,
    "description": ["Absent from the supplied session calendar. No price was invented."],
    "severity": ["medium"],
})
display(injected_gap)
assert list(injected_gap.columns) == REPORT_COLUMNS
assert "open" not in injected_gap.columns


DEMO_A is missing 1 trading days between 2024-01-02 and 2024-01-05: [datetime.date(2024, 1, 3)]


,ticker,issue_type,issue_date,description,severity
0,DEMO_A,missing_session,2024-01-03,Absent from the supplied session calendar. No ...,medium


### Outliers and inconsistent bars

Returns use the raw close, not adjusted close. The screen is `abs(return - mean) > 3 * std` inside each ticker, with the sample standard deviation (`ddof=1`) and both tails. Undefined returns stay out of the mean, the standard deviation, and the flags. If the standard deviation is zero or undefined, that ticker gets no statistical flags and a low-severity note instead.

`high < close` and `low > close` are reported on their own and again inside the lecture's OHLC-bounds check when that check also matches. Several issues on one observation stay on several report rows. Adjusted close is checked for missing, non-finite, and nonpositive values only.


In [10]:
price_issues = price_issue_report(diagnosed)
duplicate_issues = duplicate_issue_rows(exact_rows, conflicts)
quality_report_before_repair = concat_issues([price_issues, gap_issues, duplicate_issues])
assert set(quality_report_before_repair["issue_type"]).issubset(SEVERITY)

returns = simple_close_returns(diagnosed)
return_screen = return_screen_status(diagnosed, returns)
print("3σ screen status:")
display(return_screen["status"].value_counts(dropna=False).rename("tickers").to_frame())
print(
    "Tickers where the 3σ screen was not applied:",
    int(return_screen["status"].ne("applied").sum()),
)

issue_counts = (
    quality_report_before_repair.groupby(["issue_type", "severity"], dropna=False)
    .size()
    .rename("issues")
    .reset_index()
    .sort_values(["severity", "issue_type"], kind="mergesort")
    .reset_index(drop=True)
)
display(issue_counts)
samples = quality_report_before_repair.groupby("issue_type", group_keys=False).head(2)
display(samples)

bound_keys = quality_report_before_repair.loc[
    quality_report_before_repair["issue_type"].eq("ohlc_bounds"),
    ["ticker", "issue_date"],
]
bound_rows = diagnosed.merge(bound_keys, left_on=KEY_COLUMNS, right_on=["ticker", "issue_date"], how="inner")
print("Diagnosed rows with an OHLC-bounds flag:")
display(bound_rows.drop(columns="issue_date"))
print("Pre-repair diagnosis rows:", f"{len(quality_report_before_repair):,}")


3σ screen status:


,tickers
status,
applied,30


Tickers where the 3σ screen was not applied: 0


,issue_type,severity,issues
0,nonpositive_price,high,5
1,ohlc_bounds,high,5
2,exact_duplicate,low,58920
3,extreme_return,medium,5056
4,missing_adj_close,medium,18374
5,zero_volume,medium,75


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,Simple close return is more than 3 sample stan...,medium
1,AAPL,exact_duplicate,1981-01-02,Exact duplicate after ticker normalization. Ex...,low
2,AAPL,exact_duplicate,1981-01-02,Exact duplicate after ticker normalization. Ex...,low
5,AAPL,missing_adj_close,1981-01-08,"adj_close is missing. This is a missing cell, ...",medium
6,AAPL,missing_adj_close,1981-01-13,"adj_close is missing. This is a missing cell, ...",medium
18,AAPL,extreme_return,1981-04-02,Simple close return is more than 3 sample stan...,medium
6449,AXP,zero_volume,1972-12-14,Volume is zero. This can indicate a stale prin...,medium
19342,CVX,nonpositive_price,1962-01-02,open is nonpositive.,high
19343,CVX,ohlc_bounds,1962-01-02,Raw bar violates low <= open <= high or low <=...,high
31406,HON,zero_volume,1982-09-23,Volume is zero. This can indicate a stale prin...,medium


Diagnosed rows with an OHLC-bounds flag:


,ticker,ts,open,high,low,close,adj_close,volume
0,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840
1,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0
2,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800
3,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830
4,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000


Pre-repair diagnosis rows: 82,435


### Injected detector and repair checks

These examples cover required cases with no rows in the pre-repair report. Each one is detected, repaired with the lecture's rule where a repair is defined, and then checked again. A missing session is not given a price. An extreme return on a valid bar is kept. A conflicting key is quarantined rather than reduced to its first row.


In [11]:
def bar(ticker, ts, open_, high, low, close, adj_close=None, volume=1_000):
    return {
        "ticker": ticker,
        "ts": pd.Timestamp(ts),
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "adj_close": close if adj_close is None else adj_close,
        "volume": volume,
    }


def show_counts(label: str, report: pd.DataFrame) -> None:
    counts = report.groupby("issue_type").size() if not report.empty else pd.Series(dtype=int)
    print(label)
    display(counts.rename("issues").to_frame() if not counts.empty else empty_report())


def compare_close_repair(label: str, frame: pd.DataFrame) -> pd.DataFrame:
    before = price_issue_report(frame)
    repaired = repair_invalid_closes(frame)
    after = price_issue_report(repaired)
    print(f"\n{label}")
    show_counts("Before repair", before)
    display(repaired)
    show_counts("After repair", after)
    return repaired


# Lecture illustration: yesterday's close is outside today's range, so forward fill alone is not valid.
range_example = pd.DataFrame({"low": [99.0, 109.0], "high": [101.0, 111.0], "close": [100.0, np.nan]})
range_example["ffill_candidate"] = range_example["close"].ffill()
range_example["candidate_in_range"] = range_example["ffill_candidate"].between(range_example["low"], range_example["high"])
display(range_example)
assert not bool(range_example.loc[1, "candidate_in_range"])

high_below_close = pd.DataFrame([
    bar("DEMO", "2024-01-02", 10, 11, 9, 10),
    bar("DEMO", "2024-01-03", 10, 11, 9, 12),
])
repaired_high = compare_close_repair("Injected high < close. Previous close is inside today's range.", high_below_close)
assert repaired_high.loc[1, "close"] == 10
assert price_issue_report(repaired_high).query("issue_type == 'high_below_close'").empty

low_above_close = pd.DataFrame([
    bar("DEMO", "2024-01-02", 100, 101, 99, 100),
    bar("DEMO", "2024-01-03", 110, 111, 109, 108),
])
repaired_low = compare_close_repair("Injected low > close. Previous close is outside today's range, so the midpoint is used.", low_above_close)
assert repaired_low.loc[1, "close"] == 110
assert price_issue_report(repaired_low).query("issue_type == 'low_above_close'").empty

nonfinite_close = pd.DataFrame([
    bar("DEMO", "2024-01-02", 10, 11, 9, 10),
    bar("DEMO", "2024-01-03", 10, 11, 9, np.inf),
])
repaired_nonfinite = compare_close_repair("Injected non-finite close.", nonfinite_close)
assert np.isfinite(repaired_nonfinite.loc[1, "close"])
assert price_issue_report(repaired_nonfinite).query("issue_type == 'nonfinite_price'").empty

nonpositive_adjusted = pd.DataFrame([
    bar("DEMO", "2024-01-02", 10, 11, 9, 10, adj_close=-1),
])
repaired_adjusted = compare_close_repair("Injected nonpositive adjusted close. It is not rewritten from the raw close.", nonpositive_adjusted)
assert pd.isna(repaired_adjusted.loc[0, "adj_close"])
assert bool(repaired_adjusted.loc[0, "adj_close_quarantined"])
assert "nonpositive_price" not in set(price_issue_report(repaired_adjusted)["issue_type"])

missing_close = pd.DataFrame([
    bar("DEMO", "2024-01-02", 10, 11, 9, 10),
    bar("DEMO", "2024-01-03", 10, 11, 9, np.nan),
])
repaired_missing = compare_close_repair("Injected missing raw close.", missing_close)
assert repaired_missing.loc[1, "close"] == 10

missing_volume = pd.DataFrame([
    bar("DEMO", "2024-01-02", 10, 11, 9, 10, volume=np.nan),
])
repaired_volume = compare_close_repair("Injected missing volume. The fill is 0, with a repair flag.", missing_volume)
assert repaired_volume.loc[0, "volume"] == 0
assert bool(repaired_volume.loc[0, "volume_repaired"])

constant_prices = pd.DataFrame([
    bar("FLAT", f"2024-01-0{day}", 10, 11, 9, 10) for day in (2, 3, 4)
])
constant_report = price_issue_report(constant_prices)
print("\nInjected zero standard deviation")
display(constant_report)
assert constant_report.query("issue_type == 'extreme_return'").empty
assert constant_report["issue_type"].eq("undefined_return_screen").any()

single_price = pd.DataFrame([bar("SHORT", "2024-01-02", 10, 11, 9, 10)])
single_report = price_issue_report(single_price)
print("\nInjected undefined standard deviation")
display(single_report)
assert single_report.query("issue_type == 'extreme_return'").empty
assert single_report["issue_type"].eq("undefined_return_screen").any()

valid_jump_rows = [bar("JUMP", "2024-01-02", 10, 10.2, 9.8, 10)]
price = 10.0
for offset in range(1, 30):
    price *= 1.001
    valid_jump_rows.append(bar("JUMP", pd.Timestamp("2024-01-02") + pd.offsets.BDay(offset), price, price * 1.01, price * 0.99, price))
price *= 1.5
valid_jump_rows.append(bar("JUMP", pd.Timestamp("2024-01-02") + pd.offsets.BDay(30), price, price * 1.01, price * 0.99, price))
valid_jump = pd.DataFrame(valid_jump_rows)
jump_before = price_issue_report(valid_jump)
jump_after = price_issue_report(repair_invalid_closes(valid_jump))
print("\nInjected large return on a valid bar. The repair does not delete it.")
print("Extreme flags before:", int(jump_before["issue_type"].eq("extreme_return").sum()))
print("Extreme flags after:", int(jump_after["issue_type"].eq("extreme_return").sum()))
assert jump_before["issue_type"].eq("extreme_return").any()
assert jump_after["issue_type"].eq("extreme_return").any()

conflict_frame = pd.DataFrame([
    bar("DEMO", "2024-01-02", 10, 11, 9, 10),
    bar("DEMO", "2024-01-02", 10, 11, 9, 12),
])
_, conflict_only, conflict_diagnosed, _ = split_duplicate_keys(attach_labels(conflict_frame))
conflict_report = duplicate_issue_rows(pd.DataFrame(columns=VALUE_COLUMNS), conflict_only)
print("\nInjected conflicting key")
display(conflict_report)
assert len(conflict_only) == 2
assert conflict_diagnosed.empty
assert not conflict_diagnosed["close"].eq(10).any()
print("Both conflicting rows were quarantined. The close of 10 was not selected just because it came first.")


,low,high,close,ffill_candidate,candidate_in_range
0,99.0,101.0,100.0,100.0,True
1,109.0,111.0,NaN,100.0,False



Injected high < close. Previous close is inside today's range.
Before repair


,issues
issue_type,
high_below_close,1
ohlc_bounds,1
undefined_return_screen,1


,ticker,ts,open,high,low,close,adj_close,volume,close_repaired,close_quarantined,adj_close_quarantined,volume_repaired
0,DEMO,2024-01-02,10,11,9,10.0,10.0,1000,False,False,False,False
1,DEMO,2024-01-03,10,11,9,10.0,12.0,1000,True,False,False,False


After repair


...


Injected undefined standard deviation


,ticker,issue_type,issue_date,description,severity
0,SHORT,undefined_return_screen,2024-01-02,"Fewer than two defined simple returns, so the ...",low



Injected large return on a valid bar. The repair does not delete it.
Extreme flags before: 1
Extreme flags after: 1

Injected conflicting key


,ticker,issue_type,issue_date,description,severity
0,DEMO,conflicting_key,2024-01-02,The same normalized ticker and date disagree. ...,high
1,DEMO,conflicting_key,2024-01-02,The same normalized ticker and date disagree. ...,high


Both conflicting rows were quarantined. The close of 10 was not selected just because it came first.


## 1.3 Clean the data

This section repairs the delivered rows and writes `ohlc_hw_cleaned` to the homework working copy, `Homeworks/HW2/Dhruv_Gupta/data/data.db`. The course database and the `ohlc_hw` table stay unchanged. Dates are stored as `YYYY-MM-DD`, which is the form the report SQL filters expect.

The rules follow Lecture 4 module 1 (`1_visualize_and_clean.ipynb`) and the detectors from section 1.2.

**Missing raw prices.** A row whose open, high, low, and close are all missing is dropped before any fill. A remaining missing price is handled inside one ticker, in date order. When today's low and high are a valid range, the lecture's rule is used: the previous close if it sits inside that range, otherwise the midpoint of today's low and high. A missing price with no usable range is forward-filled from the last valid price of the same ticker. A fill that is nonpositive or that breaks the bar is discarded. A leading gap with no earlier price is quarantined. Nothing is taken from a later date or from another ticker.

**Impossible bars.** The documented choice is Option 2, then Option 3. Option 2 uses the previous close when that close is positive and inside the prices that are still valid. Option 3 uses the other prices on the same bar: the midpoint of low and high, or the open/close span when high or low is the broken side. Option 1, unconditional forward-fill, is not used. Lecture 4 shows that yesterday's close can fall outside today's range, and section 1.2 already demonstrates that case.

On this delivery the only broken bars are five opens recorded as 0, on the first session of CVX, JNJ, MCD, MRK, and PG. High, low, and close on those rows are positive and already satisfy `low <= close <= high`. There is no previous close. The open is set to the midpoint of that day's low and high and flagged `open_repaired`. The supplied `ohlc` table records the same open of 0, so the midpoint is an estimate required by the nonpositive-price check, not a second vendor print.

**Volume.** Missing, negative, or non-finite volume becomes 0 with `volume_repaired`. That 0 means the volume is unknown. An observed 0 stays 0, with `zero_volume_flag`, and `volume_repaired` stays false.

**Adjusted close.** The factor is `ohlc.adj_close / ohlc.close` on the same normalized ticker and the same date, and only when both reference prices are finite and positive. The repaired value is the validated raw close times that same-day factor. A missing adjusted close uses this factor. A delivered adjusted close whose factor differs from the same-day reference factor by more than 50% is rebuilt the same way. In this file those damaged factors are several times the reference factor, and they are not the previous or the next reference factor, so they are not a one-day lag across a corporate action. A smaller disagreement would be quarantined rather than overwritten. The factor is never carried forward from an earlier date. Raw open, high, low, close, and volume are not copied from `ohlc`.

**Extreme returns.** The 3σ screen remains a review flag. A close is interpolated only when the error is confirmed and the missing point sits between two valid neighbors of the same ticker. The helper below enforces `limit_area='inside'`. On this delivery the extreme closes match `ohlc`, and the large raw moves do not have a small adjusted move, so no split is identified. The three large zero-volume reversals are kept with an explicit uncertainty note. No news source was checked.

**Keys.** Tickers are normalized before duplicates are judged. Extra exact copies are removed. Conflicting `(ticker, ts)` rows are quarantined together. `ticker_raw` keeps the surviving delivered label.

Raw bounds are checked again after the same-bar repair and after the forward-fill. Adjusted-close factors are checked against the same-day reference factor after the rebuild.


In [12]:
import shutil

# Gaps wider than this are rebuilt from the same-day ohlc factor.
# The damaged prints in this file are several times larger.
# A smaller positive gap is quarantined instead of overwritten.
FACTOR_GAP_REPAIR = 0.50
FACTOR_GAP_IGNORE = 1e-4

def _valid_price(series: pd.Series) -> pd.Series:
    finite = np.isfinite(series.to_numpy(dtype=float, na_value=np.nan))
    return pd.Series(finite, index=series.index) & series.gt(0)


def _append_method(existing: pd.Series, text: str) -> pd.Series:
    return pd.Series(
        np.where(existing.eq(""), text, existing.astype(str) + ";" + text),
        index=existing.index,
    )


def collapse_delivered_rows(frame: pd.DataFrame) -> tuple[pd.DataFrame, int, pd.DataFrame]:
    """Normalize tickers, drop extra exact copies, and quarantine conflicting keys.

    A plain ticker label is kept ahead of an HTML label when both copies agree.
    """
    labeled = frame.copy()
    labeled["ticker_raw"] = labeled["ticker"].astype("string")
    labeled["ticker"] = labeled["ticker_raw"].map(normalize_ticker)
    labeled["ts"] = pd.to_datetime(labeled["ts"])
    labeled["_plain"] = labeled["ticker_raw"].astype(str).eq(labeled["ticker"])
    labeled = labeled.sort_values(
        ["ticker", "ts", "_plain", "ticker_raw"],
        ascending=[True, True, False, True],
        kind="mergesort",
    )
    extra_mask = labeled.duplicated(VALUE_COLUMNS, keep="first")
    removed_exact = int(extra_mask.sum())
    deduped = labeled.loc[~extra_mask].drop(columns="_plain")
    conflict_mask = deduped.duplicated(KEY_COLUMNS, keep=False)
    conflicts = deduped.loc[conflict_mask].copy()
    conflicts["quarantine_reason"] = "conflicting_key"
    kept = (
        deduped.loc[~conflict_mask]
        .sort_values(KEY_COLUMNS, kind="mergesort")
        .reset_index(drop=True)
    )
    return kept, removed_exact, conflicts.reset_index(drop=True)


def drop_all_raw_missing(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Drop rows with no raw OHLC at all, before any fill can borrow a neighbor."""
    missing = frame[RAW_PRICE_COLUMNS].isna().all(axis=1)
    removed = frame.loc[missing].copy()
    removed["quarantine_reason"] = "removed_all_raw_prices_missing"
    kept = frame.loc[~missing].reset_index(drop=True)
    return kept, removed.reset_index(drop=True)


def repair_raw_price_fields(frame: pd.DataFrame) -> pd.DataFrame:
    """Repair impossible raw fields with Option 2, then Option 3.

    Option 2 uses the previous close when that close is positive and inside the
    range still supported by today's other prices. Option 3 uses today's other
    prices: the midpoint of low and high, or the open/close span when high or
    low is the broken side. Option 1, unconditional forward-fill, is not used.
    """
    work = frame.sort_values(KEY_COLUMNS, kind="mergesort").reset_index(drop=True).copy()
    for column in RAW_PRICE_COLUMNS:
        work[f"{column}_repaired"] = False
    work["raw_repair_method"] = ""

    previous_close = work["close"].where(_valid_price(work["close"])).groupby(work["ticker"]).shift(1)
    low_high_ok = _valid_price(work["low"]) & _valid_price(work["high"]) & work["low"].le(work["high"])
    midpoint = (work["high"] + work["low"]) / 2

    def apply_repair(mask: pd.Series, column: str, method: str, replacement: pd.Series) -> None:
        if not bool(mask.any()):
            return
        work.loc[mask, column] = replacement.loc[mask].to_numpy()
        work.loc[mask, f"{column}_repaired"] = True
        work.loc[mask, "raw_repair_method"] = _append_method(
            work.loc[mask, "raw_repair_method"],
            f"{column}:{method}",
        ).to_numpy()

    close_outside = ~work["close"].between(work["low"], work["high"])
    close_bad = ~_valid_price(work["close"]) | close_outside.fillna(False)
    close_fixable = low_high_ok & close_bad
    previous_fits_close = close_fixable & previous_close.between(work["low"], work["high"])
    apply_repair(previous_fits_close, "close", "previous_close", previous_close)
    apply_repair(close_fixable & ~previous_fits_close, "close", "same_bar_midpoint", midpoint)

    open_outside = ~work["open"].between(work["low"], work["high"])
    open_bad = ~_valid_price(work["open"]) | open_outside.fillna(False)
    close_inside = _valid_price(work["close"]) & work["close"].between(work["low"], work["high"])
    open_fixable = low_high_ok & close_inside.fillna(False) & open_bad
    previous_fits_open = open_fixable & previous_close.between(work["low"], work["high"])
    apply_repair(previous_fits_open, "open", "previous_close", previous_close)
    apply_repair(open_fixable & ~previous_fits_open, "open", "same_bar_midpoint", midpoint)

    open_close_ok = _valid_price(work["open"]) & _valid_price(work["close"])
    span_low = work[["open", "close"]].min(axis=1)
    span_high = work[["open", "close"]].max(axis=1)
    high_bad = ~_valid_price(work["high"]) | work["high"].lt(span_high)
    low_bad = ~_valid_price(work["low"]) | work["low"].gt(span_low)
    bounds_fail = high_bad.fillna(False) | low_bad.fillna(False) | work["high"].lt(work["low"]).fillna(False)
    fix_range = open_close_ok & bounds_fail

    previous_as_high = fix_range & high_bad.fillna(False) & previous_close.ge(span_high)
    apply_repair(previous_as_high, "high", "previous_close", previous_close)
    apply_repair(fix_range & high_bad.fillna(False) & ~previous_as_high, "high", "same_bar_other_prices", span_high)

    previous_as_low = (
        fix_range
        & low_bad.fillna(False)
        & previous_close.gt(0)
        & previous_close.le(span_low)
        & previous_close.le(work["high"])
    )
    apply_repair(previous_as_low, "low", "previous_close", previous_close)
    apply_repair(fix_range & low_bad.fillna(False) & ~previous_as_low, "low", "same_bar_other_prices", span_low)
    return work


def forward_fill_missing_prices(frame: pd.DataFrame) -> pd.DataFrame:
    """Carry the last valid raw price within a ticker. Leave a broken fill blank.

    `limit` is not required: each column is filled independently, then a fill that
    is nonpositive or outside the row's remaining range is discarded. Nothing is
    filled from a later date or from another ticker.
    """
    work = frame.sort_values(KEY_COLUMNS, kind="mergesort").reset_index(drop=True).copy()
    for column in RAW_PRICE_COLUMNS:
        invalid = ~_valid_price(work[column])
        work.loc[invalid, column] = np.nan
        carried = work.groupby("ticker", sort=False)[column].ffill()
        accept = work[column].isna() & _valid_price(carried)
        if column in ("open", "close"):
            bounded = _valid_price(work["low"]) & _valid_price(work["high"]) & work["low"].le(work["high"])
            in_range = carried.between(work["low"], work["high"])
            accept = accept & (~bounded | in_range.fillna(False))
        elif column == "high":
            floor = work[["open", "close", "low"]].max(axis=1, skipna=True)
            known_floor = work[["open", "close", "low"]].apply(_valid_price).any(axis=1)
            accept = accept & (~known_floor | carried.ge(floor).fillna(False))
        else:
            cap = work[["open", "close", "high"]].min(axis=1, skipna=True)
            known_cap = work[["open", "close", "high"]].apply(_valid_price).any(axis=1)
            accept = accept & (~known_cap | carried.le(cap).fillna(False))
        if bool(accept.any()):
            work.loc[accept, column] = carried.loc[accept].to_numpy()
            work.loc[accept, f"{column}_repaired"] = True
            work.loc[accept, "raw_repair_method"] = _append_method(
                work.loc[accept, "raw_repair_method"],
                f"{column}:forward_fill",
            ).to_numpy()
    return work


def repair_volume(frame: pd.DataFrame) -> pd.DataFrame:
    """Unknown volume becomes a flagged 0. An observed 0 stays an observed 0."""
    work = frame.copy()
    volume = work["volume"]
    finite = pd.Series(np.isfinite(volume.to_numpy(dtype=float, na_value=np.nan)), index=volume.index)
    unknown = volume.isna() | (volume.notna() & (~finite | volume.lt(0)))
    work["volume_repaired"] = unknown
    work.loc[unknown, "volume"] = 0
    work["zero_volume_flag"] = work["volume"].eq(0) & ~work["volume_repaired"]
    return work


def reference_factors(reference: pd.DataFrame) -> pd.DataFrame:
    """Date-specific factor from supplied ohlc. Duplicate reference keys are rejected."""
    factors = reference.copy()
    factors["ticker"] = factors["ticker"].map(normalize_ticker)
    factors["ts"] = pd.to_datetime(factors["ts"])
    if factors.duplicated(KEY_COLUMNS).any():
        raise ValueError("Supplied ohlc has duplicate (ticker, ts) keys; no factor was applied.")
    factors = factors.rename(columns={"close": "ref_close", "adj_close": "ref_adj_close"})
    trusted = (
        _valid_price(factors["ref_close"])
        & _valid_price(factors["ref_adj_close"])
    )
    factors["ref_factor"] = np.where(trusted, factors["ref_adj_close"] / factors["ref_close"], np.nan)
    return factors[KEY_COLUMNS + ["ref_close", "ref_adj_close", "ref_factor"]]


def repair_adjusted_close(frame: pd.DataFrame, factors: pd.DataFrame) -> pd.DataFrame:
    """Rebuild adjusted close from the same-day ohlc factor. Do not carry a factor forward."""
    work = frame.merge(factors, on=KEY_COLUMNS, how="left")
    delivered_factor = work["adj_close"] / work["close"]
    comparable = _valid_price(work["close"]) & work["ref_factor"].notna() & np.isfinite(delivered_factor)
    relative_gap = (delivered_factor - work["ref_factor"]).abs() / work["ref_factor"]
    missing_or_invalid = ~_valid_price(work["adj_close"])
    damaged = comparable & relative_gap.gt(FACTOR_GAP_REPAIR)
    unresolved_gap = comparable & relative_gap.gt(FACTOR_GAP_IGNORE) & ~damaged
    can_rebuild = work["ref_factor"].notna() & _valid_price(work["close"])
    rebuild = can_rebuild & (missing_or_invalid | damaged)

    work["adj_close_repaired"] = False
    work["adjustment_factor_source"] = np.where(
        _valid_price(work["adj_close"]),
        "delivered",
        "",
    )
    work["adjustment_gap_unresolved"] = unresolved_gap.fillna(False)
    work.loc[rebuild, "adj_close"] = (work.loc[rebuild, "close"] * work.loc[rebuild, "ref_factor"]).to_numpy()
    work.loc[rebuild, "adj_close_repaired"] = True
    work.loc[rebuild & missing_or_invalid, "adjustment_factor_source"] = "ohlc_same_day_missing"
    work.loc[rebuild & damaged & ~missing_or_invalid, "adjustment_factor_source"] = "ohlc_same_day_inconsistent"
    return work


def raw_bounds_broken(frame: pd.DataFrame) -> pd.Series:
    prices_ok = pd.Series(True, index=frame.index)
    for column in RAW_PRICE_COLUMNS:
        prices_ok = prices_ok & _valid_price(frame[column])
    bounds = (
        frame["high"].lt(frame[["open", "close"]].max(axis=1))
        | frame["low"].gt(frame[["open", "close"]].min(axis=1))
        | frame["high"].lt(frame["low"])
    )
    return ~prices_ok | bounds.fillna(True)


def attach_extreme_flags(frame: pd.DataFrame) -> pd.DataFrame:
    """Flag the 3σ screen. Interpolation is a separate, opt-in step and is not applied here."""
    work = frame.sort_values(KEY_COLUMNS, kind="mergesort").reset_index(drop=True).copy()
    valid_close = work["close"].where(_valid_price(work["close"]))
    previous = valid_close.groupby(work["ticker"]).shift(1)
    returns = ((valid_close - previous) / previous).where(lambda series: np.isfinite(series))
    grouped = returns.groupby(work["ticker"])
    mean = grouped.transform("mean")
    std = grouped.transform("std")
    extreme = returns.notna() & std.gt(0) & (returns - mean).abs().gt(3 * std)
    work["extreme_return_flag"] = extreme.fillna(False)

    valid_adj = work["adj_close"].where(_valid_price(work["adj_close"]))
    previous_adj = valid_adj.groupby(work["ticker"]).shift(1)
    adj_returns = ((valid_adj - previous_adj) / previous_adj).where(lambda series: np.isfinite(series))
    next_return = returns.groupby(work["ticker"]).shift(-1)
    reversal = (
        extreme
        & returns.abs().gt(0.4)
        & work["volume"].eq(0)
        & next_return.notna()
        & (np.sign(next_return) != np.sign(returns))
        & next_return.abs().gt(returns.abs() * 0.4)
    )
    split_like = extreme & returns.abs().gt(0.4) & adj_returns.abs().lt(0.2)
    work["split_like_flag"] = split_like.fillna(False)
    note = pd.Series("", index=work.index, dtype="object")
    kept_note = (
        "Kept. Simple return is outside this ticker's 3 sample-standard-deviation band. "
        "The raw close matches supplied ohlc on this date, so the move is not an ohlc_hw-only error. "
        "Not interpolated."
    )
    reversal_note = (
        "Kept, with uncertainty. Volume is zero and the next session reverses a large part of the move. "
        "The same close is in supplied ohlc. No corporate-action filing or news source was checked, "
        "so this is not labeled a split or a news event. Not interpolated."
    )
    note.loc[work["extreme_return_flag"]] = kept_note
    note.loc[reversal.fillna(False)] = reversal_note
    work["extreme_return_note"] = note
    work.attrs["split_like_count"] = int(split_like.fillna(False).sum())
    work.attrs["reversal_count"] = int(reversal.fillna(False).sum())
    return work


def interpolate_confirmed_errors(frame: pd.DataFrame, column: str, confirmed: pd.Series) -> pd.Series:
    """Fill confirmed errors from neighbors inside the same ticker. Do not extrapolate."""
    ordered = frame.sort_values(KEY_COLUMNS, kind="mergesort")
    values = ordered[column].mask(confirmed.reindex(ordered.index).fillna(False))
    filled = values.groupby(ordered["ticker"], sort=False).transform(
        lambda series: series.interpolate(method="linear", limit_area="inside")
    )
    result = ordered[column].where(~confirmed.reindex(ordered.index).fillna(False), filled)
    return result.reindex(frame.index)


def unresolved_mask(frame: pd.DataFrame) -> pd.Series:
    adj_bad = ~_valid_price(frame["adj_close"])
    volume_bad = ~np.isfinite(frame["volume"]) | frame["volume"].lt(0)
    gap = frame.get("adjustment_gap_unresolved", pd.Series(False, index=frame.index)).fillna(False)
    no_adj_source = adj_bad & frame["adjustment_factor_source"].eq("")
    return raw_bounds_broken(frame) | adj_bad | volume_bad | gap | no_adj_source


def quarantine_reason(frame: pd.DataFrame, mask: pd.Series) -> pd.Series:
    reasons = pd.Series("", index=frame.index, dtype="object")
    broken = raw_bounds_broken(frame)
    adj_bad = ~_valid_price(frame["adj_close"])
    reasons.loc[mask & broken] = "unresolved_raw_price"
    gap = frame.get("adjustment_gap_unresolved", pd.Series(False, index=frame.index)).fillna(False)
    reasons.loc[mask & gap & ~broken] = "unresolved_adjustment_gap"
    reasons.loc[mask & adj_bad & ~broken & ~gap] = "unresolved_adjusted_close"
    return reasons.loc[mask]


def clean_market_data(
    delivered: pd.DataFrame,
    reference: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Clean one delivery. The input frames are not modified."""
    kept, removed_exact, conflicts = collapse_delivered_rows(delivered)
    kept, removed_blank = drop_all_raw_missing(kept)
    repaired = repair_raw_price_fields(kept)
    raw_failures_after_same_bar = int(raw_bounds_broken(repaired).sum())
    repaired = forward_fill_missing_prices(repaired)
    raw_failures_after_fill = int(raw_bounds_broken(repaired).sum())
    repaired = repair_volume(repaired)
    factors = reference_factors(reference)
    repaired = repair_adjusted_close(repaired, factors)
    unresolved = unresolved_mask(repaired)
    quarantine_rows = repaired.loc[unresolved].copy()
    quarantine_rows["quarantine_reason"] = quarantine_reason(repaired, unresolved).to_numpy()
    retained = repaired.loc[~unresolved].drop(
        columns=["ref_close", "ref_adj_close", "ref_factor", "adjustment_gap_unresolved"],
        errors="ignore",
    )
    retained = attach_extreme_flags(retained)
    quarantine = pd.concat(
        [
            conflicts,
            quarantine_rows,
        ],
        ignore_index=True,
        sort=False,
    )
    summary = {
        "delivered_rows": int(len(delivered)),
        "removed_exact_duplicate": removed_exact,
        "removed_all_raw_prices_missing": int(len(removed_blank)),
        "quarantined_conflicting_key": int(len(conflicts)),
        "quarantined_unresolved": int(unresolved.sum()),
        "retained_rows": int(len(retained)),
        "raw_bounds_failures_after_same_bar_repair": raw_failures_after_same_bar,
        "raw_bounds_failures_after_forward_fill": raw_failures_after_fill,
        "open_repaired": int(retained["open_repaired"].sum()) if len(retained) else 0,
        "close_repaired": int(retained["close_repaired"].sum()) if len(retained) else 0,
        "adj_close_repaired": int(retained["adj_close_repaired"].sum()) if len(retained) else 0,
        "volume_repaired": int(retained["volume_repaired"].sum()) if len(retained) else 0,
        "zero_volume_flag": int(retained["zero_volume_flag"].sum()) if len(retained) else 0,
        "extreme_return_flag": int(retained["extreme_return_flag"].sum()) if len(retained) else 0,
        "split_like_rows": int(retained.attrs.get("split_like_count", retained["split_like_flag"].sum())) if len(retained) else 0,
        "reversal_rows": int(retained.attrs.get("reversal_count", 0)) if len(retained) else 0,
    }
    summary["accounted_rows"] = (
        summary["removed_exact_duplicate"]
        + summary["removed_all_raw_prices_missing"]
        + summary["quarantined_conflicting_key"]
        + summary["quarantined_unresolved"]
        + summary["retained_rows"]
    )
    return retained, quarantine, summary


def validate_retained(frame: pd.DataFrame) -> None:
    prices = frame[EQUITY_PRICE_COLUMNS]
    if prices.isna().any().any():
        raise AssertionError("Retained prices still contain missing values.")
    if not np.isfinite(prices.to_numpy(dtype=float)).all():
        raise AssertionError("Retained prices contain a non-finite value.")
    if not prices.gt(0).all().all():
        raise AssertionError("Retained prices contain a nonpositive value.")
    if frame.duplicated(KEY_COLUMNS).any():
        raise AssertionError("Retained (ticker, ts) keys are not unique.")
    if not np.isfinite(frame["volume"]).all() or frame["volume"].lt(0).any():
        raise AssertionError("Retained volume is missing, non-finite, or negative.")
    bounds = raw_bounds_broken(frame)
    if bounds.any():
        raise AssertionError(f"Retained rows still break OHLC bounds: {int(bounds.sum())}")
    if frame["split_like_flag"].any():
        raise AssertionError(
            "A large raw move has a small adjusted move. Inspect it before retaining or interpolating."
        )


### Checks on small examples

The delivered table does not contain a fully blank bar, a conflicting key, a missing volume, or a confirmed interior error. These examples exercise those rules. They also check that a factor is taken from the same date when a split changes it, and that a previous close outside today's range is not written into the bar.


In [13]:
def bar(ticker, ts, open_, high, low, close, adj_close=None, volume=100.0):
    return {
        "ticker": ticker,
        "ts": pd.Timestamp(ts),
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "adj_close": close if adj_close is None else adj_close,
        "volume": volume,
    }


toy = pd.DataFrame([
    bar("demo", "2024-01-02", 10, 11, 9, 10, adj_close=5, volume=np.nan),
    bar("demo", "2024-01-03", np.nan, np.nan, np.nan, np.nan, adj_close=np.nan, volume=5),
    bar("demo", "2024-01-04", 10, 11, 9, np.nan, adj_close=np.nan, volume=0),
    bar("demo", "2024-01-05", 20, 22, 19, 21, adj_close=np.nan, volume=8),
    bar("lead", "2024-01-02", 10, np.nan, np.nan, np.nan, adj_close=np.nan, volume=1),
    bar("demo", "2024-01-03", np.nan, np.nan, np.nan, np.nan, adj_close=np.nan, volume=5),
    bar("<b>zz</b>", "2024-02-01", 10, 11, 9, 12, adj_close=6, volume=3),
    bar("zz", "2024-02-01", 10, 11, 9, 10, adj_close=6, volume=3),
    bar("open0", "2024-03-01", 0, 4, 2, 3, adj_close=1.5, volume=9),
])
toy_reference = pd.DataFrame([
    {"ticker": "DEMO", "ts": "2024-01-02", "close": 10.0, "adj_close": 5.0},
    {"ticker": "DEMO", "ts": "2024-01-03", "close": 10.0, "adj_close": 5.0},
    {"ticker": "DEMO", "ts": "2024-01-04", "close": 10.0, "adj_close": 5.0},
    {"ticker": "DEMO", "ts": "2024-01-05", "close": 21.0, "adj_close": 21.0},
    {"ticker": "LEAD", "ts": "2024-01-02", "close": 10.0, "adj_close": 10.0},
    {"ticker": "ZZ", "ts": "2024-02-01", "close": 10.0, "adj_close": 6.0},
    {"ticker": "OPEN0", "ts": "2024-03-01", "close": 3.0, "adj_close": 1.5},
])
toy_kept, toy_quarantine, toy_summary = clean_market_data(toy, toy_reference)
print("Small-example accounting. Delivered rows equal removed + quarantined + retained.")
display(pd.Series(toy_summary).rename("rows").to_frame())
display(toy_kept[[
    "ticker", "ts", "open", "high", "low", "close", "adj_close", "volume",
    "volume_repaired", "zero_volume_flag", "raw_repair_method", "adjustment_factor_source",
]])
display(toy_quarantine[["ticker", "ts", "quarantine_reason"]])

demo = toy_kept.loc[toy_kept["ticker"].eq("DEMO")].sort_values("ts")
assert list(demo["ts"].dt.strftime("%Y-%m-%d")) == ["2024-01-02", "2024-01-04", "2024-01-05"]
assert demo.iloc[0]["volume"] == 0 and bool(demo.iloc[0]["volume_repaired"])
assert not bool(demo.iloc[0]["zero_volume_flag"])
assert demo.iloc[1]["close"] == 10
assert demo.iloc[1]["volume"] == 0 and not bool(demo.iloc[1]["volume_repaired"])
assert bool(demo.iloc[1]["zero_volume_flag"])
assert demo.iloc[2]["adj_close"] == 21
assert demo.iloc[2]["adjustment_factor_source"] == "ohlc_same_day_missing"
assert "LEAD" not in set(toy_kept["ticker"])
assert "ZZ" not in set(toy_kept["ticker"])
assert len(toy_quarantine.loc[toy_quarantine["ticker"].eq("ZZ")]) == 2
assert set(toy_quarantine.loc[toy_quarantine["ticker"].eq("ZZ"), "quarantine_reason"]) == {"conflicting_key"}
open0 = toy_kept.loc[toy_kept["ticker"].eq("OPEN0")].iloc[0]
assert open0["open"] == 3 and open0["close"] == 3 and bool(open0["open_repaired"])
assert toy_summary["accounted_rows"] == len(toy)

outside = pd.DataFrame([
    bar("edge", "2024-01-02", 10, 11, 9, 10, adj_close=10),
    bar("edge", "2024-01-03", 21, 22, 20, np.nan, adj_close=np.nan),
])
outside_reference = pd.DataFrame([
    {"ticker": "EDGE", "ts": "2024-01-02", "close": 10.0, "adj_close": 10.0},
    {"ticker": "EDGE", "ts": "2024-01-03", "close": 21.0, "adj_close": 21.0},
])
edge_kept, _, _ = clean_market_data(outside, outside_reference)
print("Previous close is 10 and today's range is 20 to 22, so the midpoint 21 is used.")
display(edge_kept[["ticker", "ts", "close", "raw_repair_method"]])
assert edge_kept.iloc[1]["close"] == 21
assert edge_kept.iloc[1]["raw_repair_method"] == "close:same_bar_midpoint"

confirmed_error = pd.DataFrame({
    "ticker": ["T"] * 5,
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08"]),
    "close": [np.nan, 10.0, np.nan, 14.0, np.nan],
})
confirmed = pd.Series([False, False, True, False, False])
filled = interpolate_confirmed_errors(confirmed_error, "close", confirmed)
print("Confirmed interior error only. The leading and trailing gaps stay missing.")
display(pd.DataFrame({"before": confirmed_error["close"], "after": filled}))
assert list(filled.isna()) == [True, False, False, False, True]
assert filled.iloc[2] == 12
print("Small-example checks passed.")


Small-example accounting. Delivered rows equal removed + quarantined + retained.


,rows
delivered_rows,9
removed_exact_duplicate,1
removed_all_raw_prices_missing,1
quarantined_conflicting_key,2
quarantined_unresolved,1
retained_rows,4
raw_bounds_failures_after_same_bar_repair,1
raw_bounds_failures_after_forward_fill,1
open_repaired,1
close_repaired,1


,ticker,ts,open,high,low,close,adj_close,volume,volume_repaired,zero_volume_flag,raw_repair_method,adjustment_factor_source
0,DEMO,2024-01-02,10.0,11.0,9.0,10.0,5.0,0.0,True,False,,delivered
1,DEMO,2024-01-04,10.0,11.0,9.0,10.0,5.0,0.0,False,True,close:previous_close,ohlc_same_day_missing
2,DEMO,2024-01-05,20.0,22.0,19.0,21.0,21.0,8.0,False,False,,ohlc_same_day_missing
3,OPEN0,2024-03-01,3.0,4.0,2.0,3.0,1.5,9.0,False,False,open:same_bar_midpoint,delivered


,ticker,ts,quarantine_reason
0,ZZ,2024-02-01,conflicting_key
1,ZZ,2024-02-01,conflicting_key
2,LEAD,2024-01-02,unresolved_raw_price


Previous close is 10 and today's range is 20 to 22, so the midpoint 21 is used.


,ticker,ts,close,raw_repair_method
0,EDGE,2024-01-02,10.0,
1,EDGE,2024-01-03,21.0,close:same_bar_midpoint


Confirmed interior error only. The leading and trailing gaps stay missing.


,before,after
0,NaN,NaN
1,10.0,10.0
2,NaN,12.0
3,14.0,14.0
4,NaN,NaN


Small-example checks passed.


### Repair the delivered table

`clean_market_data` starts from the raw `ohlc_hw` frame again, so this section does not depend on a row that section 1.2 already dropped. The accounting has to add back to the delivered row count. Section 1.2's exact-duplicate and conflicting-key counts are the check on that identity.


In [14]:
reference_conn = sqlite3.connect(db_path.resolve().as_uri() + "?mode=ro", uri=True)
reference_prices = pd.read_sql_query(
    "SELECT ticker, ts, close, adj_close FROM ohlc",
    reference_conn,
    parse_dates=["ts"],
)
reference_conn.close()

retained, quarantine, cleaning_summary = clean_market_data(ohlc_hw, reference_prices)
assert cleaning_summary["accounted_rows"] == cleaning_summary["delivered_rows"] == len(ohlc_hw)
assert cleaning_summary["removed_exact_duplicate"] == int(analysis.duplicated(VALUE_COLUMNS).sum())
assert cleaning_summary["quarantined_conflicting_key"] == len(conflicts)
assert (
    cleaning_summary["retained_rows"]
    + cleaning_summary["quarantined_unresolved"]
    + cleaning_summary["removed_all_raw_prices_missing"]
    == len(diagnosed)
)
assert cleaning_summary["raw_bounds_failures_after_same_bar_repair"] == 0
assert cleaning_summary["raw_bounds_failures_after_forward_fill"] == 0

print("Row accounting")
display(pd.Series(cleaning_summary).rename("rows").to_frame())
print(
    "Retained rows whose surviving delivered label still differs from the normalized ticker:",
    int(retained["ticker_raw"].astype(str).ne(retained["ticker"]).sum()),
)

open_before = diagnosed.loc[:, ["ticker", "ts", "open", "high", "low", "close"]].rename(
    columns={"open": "open_before", "high": "high_before", "low": "low_before", "close": "close_before"}
)
open_after = retained.loc[
    retained["open_repaired"],
    ["ticker", "ts", "open", "high", "low", "close", "raw_repair_method"],
]
open_repairs = open_after.merge(open_before, on=["ticker", "ts"], how="left")
print("Opens repaired from a recorded 0. The midpoint uses that row's own low and high.")
display(open_repairs)
assert len(open_repairs) == 5
assert np.allclose(open_repairs["open"], (open_repairs["low_before"] + open_repairs["high_before"]) / 2)
assert np.allclose(open_repairs["close"], open_repairs["close_before"])
assert open_repairs["raw_repair_method"].eq("open:same_bar_midpoint").all()

print("Adjusted-close source")
display(retained["adjustment_factor_source"].value_counts().rename("rows").to_frame())
factors = reference_factors(reference_prices)
before_factors = diagnosed.merge(factors, on=KEY_COLUMNS, how="left")
delivered_factor = before_factors["adj_close"] / before_factors["close"]
delivered_gap = (delivered_factor - before_factors["ref_factor"]).abs() / before_factors["ref_factor"]
present_gap = delivered_gap.loc[before_factors["adj_close"].notna()]
print(
    "Largest relative factor gap among delivered values at or below the 50% repair line:",
    float(present_gap.loc[present_gap.le(FACTOR_GAP_REPAIR)].max()),
)
print(
    "Smallest relative factor gap among delivered values above the 50% repair line:",
    float(present_gap.loc[present_gap.gt(FACTOR_GAP_REPAIR)].min()),
)
assert present_gap.loc[present_gap.le(FACTOR_GAP_REPAIR)].max() < FACTOR_GAP_IGNORE
assert present_gap.loc[present_gap.gt(FACTOR_GAP_REPAIR)].min() > 1

price_columns = ["open", "high", "low", "close", "adj_close", "volume"]
null_comparison = pd.DataFrame({
    "before": diagnosed[price_columns].isna().sum(),
    "after": retained[price_columns].isna().sum(),
})
print("Missing values before repair (exact copies already removed) and after repair")
display(null_comparison)

stats_comparison = pd.concat(
    {
        "before": diagnosed[price_columns].describe(),
        "after": retained[price_columns].describe(),
    },
    axis=1,
)
print("Before and after statistics")
display(stats_comparison)

after_issues = price_issue_report(retained)
issue_comparison = pd.concat(
    {
        "before": price_issues.groupby("issue_type").size(),
        "after": after_issues.groupby("issue_type").size(),
    },
    axis=1,
).fillna(0).astype(int).rename_axis("issue_type").reset_index()
print("Issue counts. Extreme returns and observed zero volume remain review flags.")
display(issue_comparison.sort_values("issue_type", kind="mergesort"))

_, gap_issues_after = session_gap_report(retained, reference_dates)
print(f"Missing sessions after repair: {len(gap_issues_after)}")
assert gap_issues_after.empty


Row accounting


,rows
delivered_rows,397711
removed_exact_duplicate,29460
removed_all_raw_prices_missing,0
quarantined_conflicting_key,0
quarantined_unresolved,0
retained_rows,368251
raw_bounds_failures_after_same_bar_repair,0
raw_bounds_failures_after_forward_fill,0
open_repaired,5
close_repaired,0


Retained rows whose surviving delivered label still differs from the normalized ticker: 36985
Opens repaired from a recorded 0. The midpoint uses that row's own low and high.


,ticker,ts,open,high,low,close,raw_repair_method,open_before,high_before,low_before,close_before
0,CVX,1962-01-02,3.270000,3.30000,3.24000,3.30000,open:same_bar_midpoint,0.0,3.30000,3.24000,3.30000
1,JNJ,1962-01-02,0.222800,0.22338,0.22222,0.22338,open:same_bar_midpoint,0.0,0.22338,0.22222,0.22338
2,MCD,1966-07-05,0.270575,0.27366,0.26749,0.26955,open:same_bar_midpoint,0.0,0.27366,0.26749,0.26955
3,MRK,1962-01-02,0.395100,0.39703,0.39317,0.39537,open:same_bar_midpoint,0.0,0.39703,0.39317,0.39537
4,PG,1962-01-02,1.430000,1.44000,1.42000,1.43000,open:same_bar_midpoint,0.0,1.44000,1.42000,1.43000


Adjusted-close source


...

Before and after statistics


before                                                                                    after  \
                open           high            low          close      adj_close        volume           open   
count  368251.000000  368251.000000  368251.000000  368251.000000  349877.000000  3.682510e+05  368251.000000   
mean       46.308766      46.771204      45.839768      46.316711      51.711536  3.095142e+07      46.308781   
std        67.143015      67.789068      66.489203      67.154690     195.001654  1.266145e+08      67.143005   
min         0.000000       0.005208       0.004801       0.005208       0.002945  0.000000e+00       0.005208   
25%         4.210000       4.250000       4.170000       4.210000       1.395997  1.731400e+06       4.210000   
50%        19.890000      20.090000      19.670000      19.900000      11.980000  4.649600e+06      19.890000   
75%        57.560000      58.160000      56.990000      57.570000      43.790000  1.102151e+07      57.560000   
max       604.260000     608.630000     597.880000     605.400000   10442.510783  9.276606e+09     604.260000   

                                                                                 
                high            low          close      adj_close        volume  
count  368251.000000  368251.000000  368251.000000  368251.000000  3.682510e+05  
mean       46.771204      45.839768      46.316711      37.274166  3.095142e+07  
std        67.789068      66.489203      67.154690      63.587588  1.266145e+08  
min         0.005208       0.004801       0.005208       0.002945  0.000000e+00  
25%         4.250000       4.170000       4.210000       1.310000  1.731400e+06  
50%        20.090000      19.670000      19.900000      11.460000  4.649600e+06  
75%        58.160000      56.990000      57.570000      41.570000  1.102151e+07  
max       608.630000     597.880000     605.400000     605.400000  9.276606e+09

Issue counts. Extreme returns and observed zero volume remain review flags.


,issue_type,before,after
0,extreme_return,5056,5056
1,missing_adj_close,18374,0
2,nonpositive_price,5,0
3,ohlc_bounds,5,0
4,zero_volume,75,75


Missing sessions after repair: 0


### Extreme returns

A simple close return is flagged when it is more than three sample standard deviations from that ticker's own mean. The screen uses `ddof=1` and both tails, as in section 1.2. It is not a deletion rule.

Two checks decide whether any flagged close is a confirmed error:

- A split-like print would move the raw close by a large amount and leave the adjusted close almost unchanged. The threshold used here is an absolute raw return above 40% together with an absolute adjusted return below 20%.
- A one-day reversal with zero volume is suspicious, but suspicion is not identification. If the same close is in the supplied `ohlc` table, rewriting it would contradict the reference. No filing or news source is available in this project, so those rows are kept and the uncertainty is written into `extreme_return_note`.

The interpolation function was checked on the five-point example above. It is not applied to these closes. An empty confirmed-error mask is the decision.


In [15]:
print(f"Extreme-return flags: {int(retained['extreme_return_flag'].sum())}")
print(f"Split-like rows: {int(retained['split_like_flag'].sum())}")
print(f"Large zero-volume reversals: {cleaning_summary['reversal_rows']}")
assert int(retained["split_like_flag"].sum()) == 0

factors = reference_factors(reference_prices)
matched = retained.merge(factors, on=KEY_COLUMNS, how="left", validate="one_to_one")
close_gap = (matched["close"] - matched["ref_close"]).abs()
factor_gap = ((matched["adj_close"] / matched["close"]) - matched["ref_factor"]).abs() / matched["ref_factor"]
print("Maximum absolute close difference versus supplied ohlc:", float(close_gap.max()))
print("Maximum relative adjusted-factor gap versus supplied ohlc:", float(factor_gap.max()))
assert close_gap.max() == 0
assert factor_gap.max() < 1e-8

reversal_mask = retained["extreme_return_note"].str.startswith("Kept, with uncertainty")
verified_note = (
    "Kept. Simple return is outside this ticker's 3 sample-standard-deviation band. "
    "The raw close equals supplied ohlc.close on this date. "
    "No row combines a raw move above 40% with an adjusted move below 20%, so no split is identified. "
    "Not interpolated."
)
uncertainty_note = (
    "Kept, with uncertainty. Volume is zero and the next session reverses a large part of the move. "
    "The same close is in supplied ohlc. No corporate-action filing or news source was checked, "
    "so this is not labeled a split or a news event. Not interpolated."
)
retained.loc[retained["extreme_return_flag"] & ~reversal_mask, "extreme_return_note"] = verified_note
retained.loc[reversal_mask, "extreme_return_note"] = uncertainty_note

reversal_view = retained.loc[
    reversal_mask,
    ["ticker", "ts", "open", "high", "low", "close", "adj_close", "volume", "extreme_return_note"],
].copy()
history = retained.loc[
    retained["ticker"].isin(reversal_view["ticker"]),
    ["ticker", "ts", "close", "adj_close", "volume"],
].sort_values(["ticker", "ts"], kind="mergesort").copy()
history["simple_return"] = history.groupby("ticker")["close"].pct_change(fill_method=None)
windows = []
for row in reversal_view.itertuples(index=False):
    windows.append(history.loc[
        history["ticker"].eq(row.ticker)
        & history["ts"].between(row.ts - pd.Timedelta(days=7), row.ts + pd.Timedelta(days=7))
    ])
neighbors = pd.concat(windows, ignore_index=True)
print("Sessions within seven calendar days of each large zero-volume reversal")
display(neighbors.sort_values(["ticker", "ts"], kind="mergesort"))
print("The flagged reversal rows")
display(reversal_view.drop(columns="extreme_return_note"))
assert len(reversal_view) == 3
assert reversal_view["volume"].eq(0).all()


Extreme-return flags: 5056
Split-like rows: 0
Large zero-volume reversals: 3


Maximum absolute close difference versus supplied ohlc: 0.0
Maximum relative adjusted-factor gap versus supplied ohlc: 0.0
Sessions within seven calendar days of each large zero-volume reversal


,ticker,ts,close,adj_close,volume,simple_return
0,TRV,1976-10-21,4.53,0.96384,188000,0.011161
1,TRV,1976-10-22,4.52,0.96051,111200,-0.002208
2,TRV,1976-10-25,4.52,0.96051,29600,0.000000
3,TRV,1976-10-26,4.50,0.95719,91200,-0.004425
4,TRV,1976-10-27,4.61,0.98046,230400,0.024444
5,TRV,1976-10-28,9.22,3.21000,0,1.000000
6,TRV,1976-10-29,4.69,0.99707,79200,-0.491323
7,TRV,1976-11-01,4.66,0.99043,59200,-0.006397
8,TRV,1976-11-03,4.42,0.94057,159200,-0.051502
9,TRV,1976-11-04,4.47,0.95054,95200,0.011312


The flagged reversal rows


,ticker,ts,open,high,low,close,adj_close,volume
318332,TRV,1976-10-28,9.22,9.22,9.22,9.22,3.21,0
318873,TRV,1978-12-20,8.03,8.03,8.03,8.03,2.80,0
319538,TRV,1981-08-10,12.06,12.06,12.06,12.06,4.20,0


### Validation and the cleaned table

The retained rows have to pass the homework checks: finite positive raw and adjusted prices, unique `(ticker, ts)` keys, finite nonnegative volume, and valid raw bounds. Review flags for extreme returns and observed zero volume are allowed to remain.

The write target is the homework copy. If that file does not exist yet, it is copied from the database this notebook read. `if_exists='replace'` is used only for `ohlc_hw_cleaned` and `ohlc_hw_quarantine`. A read-back applies the report's `BETWEEN` date filter.


In [16]:
validate_retained(retained)

COURSE_DB = REPO_ROOT / "Lectures" / "Lecture 4" / "data" / "data.db"
WRITE_DB = HOMEWORK_DIR / "data" / "data.db"
if WRITE_DB.resolve() == COURSE_DB.resolve():
    raise RuntimeError("Refusing to write the cleaned table into the course database.")
WRITE_DB.parent.mkdir(parents=True, exist_ok=True)
if not WRITE_DB.exists():
    shutil.copy2(db_path, WRITE_DB)
    print(f"Created the working copy at {WRITE_DB}")
else:
    print(f"Using the existing working copy at {WRITE_DB}")

flag_columns = [
    "open_repaired", "high_repaired", "low_repaired", "close_repaired",
    "adj_close_repaired", "volume_repaired", "zero_volume_flag", "extreme_return_flag",
]
cleaned_columns = [
    "ts", "ticker", "ticker_raw",
    "open", "high", "low", "close", "adj_close", "volume",
    *flag_columns,
    "raw_repair_method", "adjustment_factor_source", "extreme_return_note",
]
cleaned_out = retained[cleaned_columns].copy()
cleaned_out["ts"] = pd.to_datetime(cleaned_out["ts"]).dt.strftime("%Y-%m-%d")
for column in flag_columns:
    cleaned_out[column] = cleaned_out[column].astype(int)
if not np.isclose(cleaned_out["volume"], np.round(cleaned_out["volume"])).all():
    raise AssertionError("A retained volume is not a whole number.")
cleaned_out["volume"] = cleaned_out["volume"].round().astype("int64")
assert cleaned_out["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()

quarantine_columns = [
    "ticker", "ticker_raw", "ts", "open", "high", "low", "close",
    "adj_close", "volume", "quarantine_reason",
]
quarantine_out = quarantine.reindex(columns=quarantine_columns).copy()
if len(quarantine_out):
    quarantine_out["ts"] = pd.to_datetime(quarantine_out["ts"]).dt.strftime("%Y-%m-%d")

with sqlite3.connect(WRITE_DB) as write_conn:
    raw_before = pd.read_sql_query(
        "SELECT COUNT(*) AS rows, SUM(adj_close IS NULL) AS missing_adj FROM ohlc_hw",
        write_conn,
    )
    cleaned_out.to_sql("ohlc_hw_cleaned", write_conn, if_exists="replace", index=False)
    quarantine_out.to_sql("ohlc_hw_quarantine", write_conn, if_exists="replace", index=False)
    raw_after = pd.read_sql_query(
        "SELECT COUNT(*) AS rows, SUM(adj_close IS NULL) AS missing_adj FROM ohlc_hw",
        write_conn,
    )
    assert raw_before.equals(raw_after)
    readback = pd.read_sql_query(
        """
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT ticker || '|' || ts) AS keys,
               SUM(open_repaired) AS opens_repaired,
               SUM(adj_close_repaired) AS adj_repaired,
               SUM(volume_repaired) AS volumes_repaired,
               SUM(zero_volume_flag) AS zero_volume_flags,
               SUM(extreme_return_flag) AS extreme_flags,
               MIN(ts) AS first_ts,
               MAX(ts) AS last_ts
        FROM ohlc_hw_cleaned
        """,
        write_conn,
    )
    january = pd.read_sql_query(
        """
        SELECT COUNT(*) AS rows
        FROM ohlc_hw_cleaned
        WHERE ts BETWEEN '2024-01-01' AND '2024-01-31'
        """,
        write_conn,
    )
    quarantine_count = pd.read_sql_query(
        "SELECT COUNT(*) AS rows FROM ohlc_hw_quarantine",
        write_conn,
    )
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        write_conn,
    )

print("Working-copy tables")
display(tables)
print("ohlc_hw row count and missing adjusted closes, before and after the write")
display(pd.concat({"before_write": raw_before.iloc[0], "after_write": raw_after.iloc[0]}, axis=1))
print("Read-back of ohlc_hw_cleaned")
display(readback)
print("Rows with ts between 2024-01-01 and 2024-01-31")
display(january)
print("Quarantine table")
display(quarantine_count)

assert int(readback.loc[0, "rows"]) == int(readback.loc[0, "keys"]) == len(cleaned_out)
assert int(readback.loc[0, "opens_repaired"]) == 5
assert int(readback.loc[0, "volumes_repaired"]) == 0
assert int(january.loc[0, "rows"]) > 0
assert int(quarantine_count.loc[0, "rows"]) == cleaning_summary["quarantined_conflicting_key"] + cleaning_summary["quarantined_unresolved"]
assert str(readback.loc[0, "first_ts"])[:10] == readback.loc[0, "first_ts"]
assert len(readback.loc[0, "first_ts"]) == 10

with sqlite3.connect(COURSE_DB.resolve().as_uri() + "?mode=ro", uri=True) as course_conn:
    course_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        course_conn,
    )
assert "ohlc_hw_cleaned" not in set(course_tables["name"])
assert "ohlc_hw_quarantine" not in set(course_tables["name"])
print(f"Saved {len(cleaned_out):,} rows to {WRITE_DB} as ohlc_hw_cleaned.")
print("The course database does not contain ohlc_hw_cleaned.")


Created the working copy at /Users/dhruvgupta/Desktop/PythonPreProgramBerkeleyMFE/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Dhruv_Gupta/data/data.db


Working-copy tables


,name
0,market_caps
1,ohlc
2,ohlc_hw
3,ohlc_hw_cleaned
4,ohlc_hw_quarantine


ohlc_hw row count and missing adjusted closes, before and after the write


,before_write,after_write
rows,397711,397711
missing_adj,19855,19855


...

Rows with ts between 2024-01-01 and 2024-01-31


,rows
0,630


Quarantine table


,rows
0,0


Saved 368,251 rows to /Users/dhruvgupta/Desktop/PythonPreProgramBerkeleyMFE/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Dhruv_Gupta/data/data.db as ohlc_hw_cleaned.
The course database does not contain ohlc_hw_cleaned.
